# PyPSA to SMS++ conversion step by step

This notebook describes how to perform the conversion step by step: first convert the PyPSA model into a SMS++ object, then optimize it and finally read back a solved PyPSA network with the results from SMS++.

In particular:
1. Create a PyPSA network
2. Convert the PyPSA network to a SMS++ object and explore the SMS++ object
3. Optimize the SMS++ object
4. Read back the results into a PyPSA network

## 1. Creation of the PyPSA model (from UCBlock)

We first create a simple PyPSA model.

In [ ]:
import pypsa
import pandas as pd
import pypsa2smspp

#### The following code creates the desired pypsa model.

In [ ]:
n = pypsa.Network()
n.set_snapshots(pd.date_range("2024-01-01T00:00", "2024-01-01T23:00", freq="h"))

# Add carriers
n.add("Carrier", "AC")

# Add buses
n_buses = 2
for b in range(n_buses):
    n.add("Bus", f"bus{b}", carrier="AC")

# Add lines in a radial topology using bidirectional links
n_lines = n_buses - 1
for l in range(n_lines):
    n.add(
        "Link",
        f"line{l}",
        bus0=f"bus{l}",
        bus1=f"bus{l+1}",
        length=1,
        capital_cost=1000,
        p_min_pu=-1,
        p_nom_extendable=True,
    )

# Add a load to each bus
n_loads = n_buses
for l in range(n_loads):
    n.add("Load", f"load{l}", bus=f"bus{l}", p_set=pd.Series(100, index=n.snapshots))

# Add a generator to the first bus
n.add(
    "Generator",
    "gen0",
    bus="bus0",
    p_nom_extendable=True,
    capital_cost=1000,
    marginal_cost=1,
)

n_clean = n.copy()

In [ ]:
n.optimize(solver_name="highs")

## 2. Create a SMS++ object from the PyPSA model

The following code creates the SMS++ object from the PyPSA model, as a pysmspp.SMSNetwork object, and explores it.

If the user wants to use only this object or edit it before the processing, they can do so at this point. Be aware that editing the SMS++ object may lead to inconsistencies with the original PyPSA model and hence may cause issues when reading back the results into a PyPSA network.

In [ ]:
tran = pypsa2smspp.Transformation()
verbose = True
sms_network = tran.create_model(n_clean, verbose=verbose)

Showcase the pysmspp object that is created by the conversion process.

In [ ]:
sms_network

and its inner block structure using the print_tree function from pysmspp.

In [ ]:
sms_network.print_tree(show_all=True)

## 3. Execute the optimization on the transformation object

In [ ]:
tran.optimize(verbose=verbose)

## 4. Perform the conversion back to a PyPSA network and explore the results

In [ ]:
n_smspp = tran.retrieve_solution(n_clean, verbose=verbose)
n_smspp

## 5. Check the results

Retrieve PyPSA statistics

In [ ]:
n_stat = n.statistics.energy_balance(comps=["Generator"]).droplevel([0, 2])
n_stat

Retrieve SMS++ statistics

In [ ]:
n_parsed_stat = n_smspp.statistics.energy_balance(comps=["Generator"]).droplevel([0, 2])
n_parsed_stat

Calculate difference between the two

In [ ]:
error_stat = n_stat - n_parsed_stat

merged_stat = pd.concat([n_stat, n_parsed_stat, error_stat], axis=1)
merged_stat.columns = ["pypsa", "smspp", "difference"]
merged_stat